# 07 · Benchmark: ML vs Theory (CPU / GPU)

Standalone Section 3 notebook. Assumes Sections 1–2 chains already exist in
`outputs/paper/chains/` (generated by `run_cosmo.py`).

| Benchmark | Engine |
|---|---|
| ML MCMC | XGBoost surrogate, GPU booster, 1024 chains |
| Theory CPU MCMC | Exact χ², `ProcessPoolExecutor` |
| Theory GPU MCMC | Exact χ², JAX `vmap` on GPU |
| Dataset GPU vs CPU | JAX batch vs extrapolated CPU time |


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys, os, json, time, concurrent.futures, multiprocessing
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import getdist
import getdist.plots
from iminuit import Minuit

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from cosmoml.data import load_pantheon_plus, load_des_2024, load_des_2025, load_desi_bao
from cosmoml.theory import chi2_sne, chi2_sne_des, chi2_bao, chi2_joint
from cosmoml.priors import planck_prior_chi2
from cosmoml.sampling import build_chi2_dataset, load_or_build, _chi2_worker
from cosmoml.ml import (
    train_xgb, plot_learning_curve,
    shap_summary, shap_waterfall, shap_dependence_all,
    use_paper_style,
)
from cosmoml.ml.marginal import _parallel_mcmc, _render_getdist

from cosmoml.theory import make_chi2_numpy_fn


use_paper_style()
matplotlib.rcParams['text.usetex'] = False

# Paths — outputs go to ~/alejandro/cosmoml/ (outside repo)
PAPER_DIR    = Path.home() / 'alejandro' / 'cosmoml'
DATASETS_DIR = PAPER_DIR / 'datasets'
MODELS_DIR   = PAPER_DIR / 'models'
FIGURES_DIR  = PAPER_DIR / 'figures'
CHAINS_DIR   = PAPER_DIR / 'chains'
for _d in (DATASETS_DIR, MODELS_DIR, FIGURES_DIR, CHAINS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

FORCE_RETRAIN    = False
ML_MCMC_TREES   = 500   # trees used during ML MCMC (0 = all)

GLOBAL_RANGES = {
    'Om': (0.1,   0.9),
    'H0': (20.0, 100.0),
    'w0': (-3.0,  0.2),
    'wa': (-3.0,  2.0),
}
LABELS     = {'Om': r'$\Omega_m$', 'H0': r'$H_0$', 'w0': r'$w_0$', 'wa': r'$w_a$'}
MARKERS_DE = {'w0': -1.0, 'wa': 0.0}
FEATURES_4 = ['Om', 'H0', 'w0', 'wa']
RANGES_4   = GLOBAL_RANGES.copy()

## Data & χ² functions

In [2]:
panth   = load_pantheon_plus(apply_mask=True)
des2024 = load_des_2024()
des2025 = load_des_2025()
bao     = load_desi_bao()
print(f'Pantheon+ : {len(panth)} | DES2024 : {len(des2024)} | DES2025 : {len(des2025)} | BAO : {len(bao)}')


Pantheon+ : 1657 | DES2024 : 1829 | DES2025 : 1820 | BAO : 13


In [3]:
# Top-level scope required so ProcessPoolExecutor can pickle these functions

def chi2_bao_only(Om, H0, w0, wa):
    return chi2_bao(bao, Om=Om, H0=H0, w0=w0, wa=wa)

def chi2_panth_bao(Om, H0, w0, wa):
    return chi2_joint(panth, bao, Om=Om, H0=H0, w0=w0, wa=wa,
                      sne_kwargs={'use_cepheid_calibrators': False})

def chi2_panth_bao_cmb(Om, H0, w0, wa):
    return chi2_panth_bao(Om, H0, w0, wa) + planck_prior_chi2(Om=Om, H0=H0)


## Helper functions

In [4]:
def _make_gpu_predict_fn(model, n_trees=0):
    """GPU booster predict — bypasses the CPU copy inside plot_corner_marginal."""
    from cosmoml.ml.train import LogChi2Model
    if isinstance(model, LogChi2Model):
        booster = model.raw_model.get_booster().copy()
        booster.set_param({'device': 'cuda'})
        y_min = model.y_min
        irange = (0, n_trees) if n_trees > 0 else (0, 0)
        def predict_fn(arr):
            log_y = booster.inplace_predict(arr.astype(np.float32), iteration_range=irange)
            return (10.0 ** log_y) - 1.0 + y_min
    else:
        booster = model.get_booster().copy()
        booster.set_param({'device': 'cuda'})
        irange = (0, n_trees) if n_trees > 0 else (0, 0)
        def predict_fn(arr):
            return booster.inplace_predict(arr.astype(np.float32), iteration_range=irange)
    return predict_fn


def locate_bestfit(chi2_fn, features, ranges):
    init = {f: (ranges[f][0] + ranges[f][1]) / 2.0 for f in features}
    if 'w0' in features: init['w0'] = -1.0
    if 'wa' in features: init['wa'] =  0.0
    if 'Om' in features: init['Om'] =  0.3
    if 'H0' in features: init['H0'] = 68.0
    m = Minuit(chi2_fn, **init)
    m.limits = [ranges[f] for f in features]
    m.migrad()
    cov_matrix = None
    try:
        m.hesse()
        errors = {f: float(m.errors[f]) for f in features}
        if m.covariance is not None:
            cov_matrix = np.array(m.covariance)
    except Exception:
        errors = {f: 0.1 * (ranges[f][1] - ranges[f][0]) for f in features}
    REF = {f: float(m.values[f]) for f in features}
    print(f"  Best-fit : {', '.join(f'{f}={v:.4f}' for f, v in REF.items())}")
    print(f"  Errors   : {', '.join(f'{f}={errors[f]:.4f}' for f in features)}")
    print(f'  chi2_min : {m.fval:.2f}')
    if cov_matrix is not None:
        print('  Covariance: OK (correlated sampling enabled)')
    return REF, m.fval, errors, cov_matrix


def build_dataset(chi2_fn, section, features, ranges, n_gaussian=0, gaussian_sigma_scale=3.0, n_random=80_000):
    csv_path = DATASETS_DIR / f'{section}_dataset.csv'
    REF, _, errors, cov_matrix = locate_bestfit(chi2_fn, features, ranges)
    ndim = len(features)

    def builder():
        slices = []
        for _i in range(ndim):
            for _j in range(_i + 1, ndim):
                fi, fj = features[_i], features[_j]
                fixed  = {f: REF[f] for f in features if f not in (fi, fj)}
                slices.append({fi: ranges[fi], fj: ranges[fj], **fixed, '_n': 20_000})
        gclouds = None
        if n_gaussian > 0:
            if cov_matrix is not None:
                # Use full Hessian covariance — captures degeneracy directions.
                gclouds = [{'center': REF, 'cov': cov_matrix.tolist(),
                            'scale': gaussian_sigma_scale, 'n': n_gaussian,
                            'bounds': {f: ranges[f] for f in features}}]
            else:
                sigma = {f: gaussian_sigma_scale * errors[f] for f in features}
                gclouds = [{'center': REF, 'sigma': sigma, 'n': n_gaussian,
                            'bounds': {f: ranges[f] for f in features}}]
        return build_chi2_dataset(
            chi2_fn=chi2_fn, param_names=features,
            slices=slices,
            random_box={f: ranges[f] for f in features},
            n_random=n_random,
            gaussian_clouds=gclouds,
            save_to=csv_path, seed=42,
        )

    df = load_or_build(csv_path, builder, force=FORCE_RETRAIN)
    print(f'  Dataset  : {len(df):,} rows | chi2 [{df["chi2"].min():.2f}, {df["chi2"].max():.2f}]')
    return df, REF, cov_matrix


def _generate_param_array(features, ranges, ref, ndim):
    """Replicate build_chi2_dataset sampling exactly (seed=42)."""
    rng = np.random.default_rng(42)
    slices_spec = []
    for _i in range(ndim):
        for _j in range(_i + 1, ndim):
            fi, fj = features[_i], features[_j]
            fixed = {f: ref[f] for f in features if f not in (fi, fj)}
            slices_spec.append({fi: ranges[fi], fj: ranges[fj], **fixed})
    n_slice, n_random = 20_000, 80_000
    blocks = []
    for spec in slices_spec:
        block = {}
        for k, v in spec.items():
            block[k] = (rng.uniform(v[0], v[1], n_slice)
                        if isinstance(v, tuple) else np.full(n_slice, float(v)))
        blocks.append(block)
    random_block = {f: rng.uniform(ranges[f][0], ranges[f][1], n_random) for f in features}
    blocks.append(random_block)
    params = {f: np.concatenate([b[f] for b in blocks]) for f in features}
    arr = np.column_stack([params[f] for f in features]).astype(np.float32)
    return arr, params


def build_dataset_gpu(gpu_predict_fn, chi2_fn_cpu, section, features, ranges, ref):
    """Same sampling as build_dataset (seed=42), chi2 via JAX vmap GPU.
    Compares against existing CPU dataset and times a CPU sample for speedup.
    """
    from joblib.externals.loky import get_reusable_executor
    csv_gpu = DATASETS_DIR / f'{section}_dataset_gpu.csv'
    csv_cpu = DATASETS_DIR / f'{section}_dataset.csv'
    ndim    = len(features)
    n_cores = min(16, max(1, multiprocessing.cpu_count() - 1))
    arr, params = _generate_param_array(features, ranges, ref, ndim)
    total = len(arr)

    if csv_gpu.exists() and not FORCE_RETRAIN:
        print(f'  [DS-GPU]  Cache: {csv_gpu} ({total:,} pts)')
        df_gpu = pd.read_csv(csv_gpu)
        gpu_wall = None
    else:
        print(f'  [DS-GPU]  Evaluando {total:,} pts (JAX GPU batch)...')
        t0 = time.perf_counter()
        chi2_gpu = gpu_predict_fn(arr)
        gpu_wall = time.perf_counter() - t0
        print(f'  [DS-GPU]  {gpu_wall:.2f}s  ({total / gpu_wall:,.0f} pts/s)')
        df_gpu = pd.DataFrame({**params, 'chi2': chi2_gpu})[features + ['chi2']]
        df_gpu.to_csv(csv_gpu, index=False)

    N_CPU_SAMPLE = 5_000
    tasks = [
        (chi2_fn_cpu, {f: float(arr[i, j]) for j, f in enumerate(features)})
        for i in range(N_CPU_SAMPLE)
    ]
    print(f'  [DS-CPU]  Timing sample {N_CPU_SAMPLE:,} pts ({n_cores} cores)...')
    t0 = time.perf_counter()
    executor = get_reusable_executor(max_workers=n_cores)
    list(executor.map(_chi2_worker, tasks, chunksize=100))
    cpu_sample_wall = time.perf_counter() - t0
    cpu_extrap = cpu_sample_wall / N_CPU_SAMPLE * total
    speedup = (cpu_extrap / gpu_wall) if (gpu_wall and gpu_wall > 0) else None
    print(f'  [DS-CPU]  {cpu_sample_wall:.1f}s / {N_CPU_SAMPLE:,} pts → extrap full: {cpu_extrap:.0f}s')
    if speedup:
        print(f'  [DS-SPD]  Speedup GPU vs CPU (extrap): {speedup:.1f}x')

    cmp = {}
    if csv_cpu.exists():
        df_cpu = pd.read_csv(csv_cpu)
        n_cmp = min(len(df_cpu), len(df_gpu))
        diff = np.abs(df_cpu['chi2'].values[:n_cmp] - df_gpu['chi2'].values[:n_cmp])
        rel  = diff / np.abs(df_cpu['chi2'].values[:n_cmp]).clip(1e-10)
        cmp  = {'n': int(n_cmp), 'max_abs_diff': float(diff.max()),
                'mean_abs_diff': float(diff.mean()), 'max_rel_diff': float(rel.max())}
        print(f'  [DS-CMP]  max|Δχ²|={cmp["max_abs_diff"]:.4f} | mean|Δχ²|={cmp["mean_abs_diff"]:.4f}')

    meta = {
        'gpu_wall_s': round(gpu_wall, 2) if gpu_wall else None,
        'cpu_sample_s': round(cpu_sample_wall, 2),
        'cpu_extrap_s': round(cpu_extrap, 1),
        'speedup': round(speedup, 1) if speedup else None,
        'n_pts': int(total), 'comparison': cmp,
    }
    return df_gpu, meta


def train_and_shap(df, features, section, title=''):
    model, info = train_xgb(
        df, features=features, log_target=True,
        hp_overrides=dict(n_estimators=5000, learning_rate=0.03,
                          max_depth=10, device='cuda'),
        cache_path=MODELS_DIR / f'{section}_model.ubj',
        force_retrain=FORCE_RETRAIN,
    )
    plot_learning_curve(info,
        title=f'{title} — Learning Curve (R²={info["r2"]:.5f})', show=True)
    shap_v, X_s = shap_summary(model, info['X_val'],
        title=f'{title} — SHAP', save_dir=FIGURES_DIR, prefix=section, show=True)
    shap_waterfall(shap_v, idx=0,
        title=f'{title} — SHAP waterfall', show=True)
    shap_dependence_all(shap_v, X_s, save_dir=FIGURES_DIR, prefix=section, show=True)
    return model, info, shap_v, X_s


In [5]:
def _render_and_show(samples, features, section, title, markers=None,
                      suffix='', save=True):
    fig = _render_getdist(
        samples, features,
        [LABELS.get(f, f).replace('$', '') for f in features],
        markers, title, smooth_scale=0.5, ranges=RANGES_4,
    )
    if save:
        path = FIGURES_DIR / f'{section}_getdist{suffix}.png'
        fig.savefig(path, dpi=200, bbox_inches='tight')
        print(f'  Saved: {path}')
    plt.show()
    return fig


def run_mcmc_and_getdist(model, features, ranges, ref, section,
                          labels, markers=None, title='', proposal_cov=None, n_trees_mcmc=0):
    lows   = np.array([ranges[f][0] for f in features])
    highs  = np.array([ranges[f][1] for f in features])
    center = np.array([ref[f] for f in features])

    predict_fn = _make_gpu_predict_fn(model, n_trees=n_trees_mcmc)
    if n_trees_mcmc > 0:
        print(f'  [ML] Using {n_trees_mcmc} trees for MCMC')
    t0 = time.perf_counter()
    samples = _parallel_mcmc(
        predict_fn, lows, highs, center, len(features),
        n_chains=1024, n_steps=10_000, burn_in=500, seed=42, ess_target=10_000,
        proposal_cov=proposal_cov,
    )
    wall = time.perf_counter() - t0
    print(f'  [ML] Done: {wall:.1f}s  ({len(samples):,} samples)')
    meta = {'cached': False, 'wall_s': round(wall, 2), 'n_samples': len(samples)}

    _render_and_show(samples, features, section, title, markers)
    return samples, meta


def plot_getdist_comparison(samples_list, dataset_labels, features,
                             labels, markers=None, title='',
                             save_path=None, filled=None, ranges=None):
    COLORS = ['#0044cc', '#cc0000', '#009933', '#cc6600']
    n = len(samples_list)
    if filled is None:
        filled = [True] + [False] * (n - 1)
    elif isinstance(filled, bool):
        filled = [filled] * n
    str_labels = [labels.get(f, f).replace('$', '') for f in features]
    mc_ranges = {f: list(r) for f, r in ranges.items()} if ranges else None
    mc_list = [
        getdist.MCSamples(
            samples=s, names=features, labels=str_labels, label=dl,
            ranges=mc_ranges,
            settings={'smooth_scale_2D': 0.5, 'smooth_scale_1D': 0.5},
        )
        for s, dl in zip(samples_list, dataset_labels)
    ]
    g = getdist.plots.get_subplot_plotter()
    matplotlib.rcParams['text.usetex'] = False
    g.triangle_plot(
        mc_list, filled=filled, contour_colors=COLORS[:n],
        contour_lws=[2.0] * n, markers=markers,
        marker_args={'ls': '--', 'color': 'gray', 'lw': 1.5, 'alpha': 0.8}
        if markers else None,
        legend_labels=dataset_labels, legend_loc='upper right',
    )
    if ranges:
        for i, fi in enumerate(features):
            for j, fj in enumerate(features[:i + 1]):
                ax = g.subplots[i][j]
                if ax is None: continue
                ax.set_xlim(*ranges[fj])
                if i != j: ax.set_ylim(*ranges[fi])
    if title:
        g.fig.suptitle(title, fontsize=13, y=1.01)
    if save_path:
        g.fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'  Saved: {save_path}')
    plt.show()


def _make_theory_predict_fn(chi2_fn, features, n_cores):
    from joblib.externals.loky import get_reusable_executor
    executor = get_reusable_executor(max_workers=n_cores, reuse=False)
    chunksize = max(1, 1024 // (n_cores * 4))
    def predict_fn(arr):
        tasks = [
            (chi2_fn, {f: float(arr[i, j]) for j, f in enumerate(features)})
            for i in range(len(arr))
        ]
        return np.array(list(executor.map(_chi2_worker, tasks,
                                          chunksize=chunksize)), dtype=float)
    return predict_fn


def run_theory_mcmc(chi2_fn, features, ranges, ref, section,
                     labels, markers=None, title='', proposal_cov=None,
                     n_chains=1024, n_steps=10_000, ess_target=10_000):
    n_cores = min(16, max(1, multiprocessing.cpu_count() - 1))
    lows   = np.array([ranges[f][0] for f in features])
    highs  = np.array([ranges[f][1] for f in features])
    center = np.array([ref[f] for f in features])

    print(f'  [TH-CPU] Running MCMC ({n_chains} chains, {n_cores} cores)...')
    t0 = time.perf_counter()
    predict_fn = _make_theory_predict_fn(chi2_fn, features, n_cores)
    samples = _parallel_mcmc(
        predict_fn, lows, highs, center, len(features),
        n_chains=n_chains, n_steps=n_steps, burn_in=500, seed=42, ess_target=ess_target,
        proposal_cov=proposal_cov,
    )
    wall = time.perf_counter() - t0
    print(f'  [TH-CPU] Done: {wall:.1f}s  ({len(samples):,} samples)')
    meta = {'cached': False, 'wall_s': round(wall, 2), 'n_samples': len(samples)}

    _render_and_show(samples, features, section, title + ' [Theory CPU]', markers,
                      suffix='_theory')
    return samples, meta


def run_theory_mcmc_gpu(gpu_predict_fn, features, ranges, ref, section,
                         labels, markers=None, title='', proposal_cov=None):
    lows   = np.array([ranges[f][0] for f in features])
    highs  = np.array([ranges[f][1] for f in features])
    center = np.array([ref[f] for f in features])

    print('  [TH-GPU] Running MCMC (JAX/GPU)...')
    t0 = time.perf_counter()
    samples = _parallel_mcmc(
        gpu_predict_fn, lows, highs, center, len(features),
        n_chains=1024, n_steps=10_000, burn_in=500, seed=42, ess_target=10_000,
        proposal_cov=proposal_cov,
    )
    wall = time.perf_counter() - t0
    print(f'  [TH-GPU] Done: {wall:.1f}s  ({len(samples):,} samples)')
    meta = {'cached': False, 'wall_s': round(wall, 2), 'n_samples': len(samples)}

    _render_and_show(samples, features, section, title + ' [Theory GPU]', markers,
                      suffix='_theory_gpu')
    return samples, meta

def run_theory_mcmc_numpy(np_predict_fn, features, ranges, ref, section,
                           labels, markers=None, title='', proposal_cov=None):
    lows   = np.array([ranges[f][0] for f in features])
    highs  = np.array([ranges[f][1] for f in features])
    center = np.array([ref[f] for f in features])

    print('  [TH-NP] Running MCMC (numpy batched)...')
    t0 = time.perf_counter()
    samples = _parallel_mcmc(
        np_predict_fn, lows, highs, center, len(features),
        n_chains=1024, n_steps=10_000, burn_in=500, seed=42, ess_target=10_000,
        proposal_cov=proposal_cov,
    )
    wall = time.perf_counter() - t0
    print(f'  [TH-NP] Done: {wall:.1f}s  ({len(samples):,} samples)')
    meta = {'cached': False, 'wall_s': round(wall, 2), 'n_samples': len(samples)}

    _render_and_show(samples, features, section, title + ' [Theory NP]', markers,
                      suffix='_theory_np')
    return samples, meta

## Section 1 — Datasets (all 3 scenarios)

In [ ]:
SECTION_BAO = '7_bao'
TITLE_BAO   = 'w0waCDM · DESI BAO only'
SECTION_PB  = '7_pb'
TITLE_PB    = 'w0waCDM · Pantheon+ + DESI BAO'
SECTION_PBC = '7_pbc'
TITLE_PBC   = 'w0waCDM · Pantheon+ + DESI BAO + CMB'

print(f'=== {TITLE_BAO} ===')
df_bao, REF_BAO, COV_BAO = build_dataset(
    chi2_bao_only, SECTION_BAO, FEATURES_4, RANGES_4,
    n_gaussian=150_000, gaussian_sigma_scale=3.0,
)
print(f'\n=== {TITLE_PB} ===')
df_pb, REF_PB, COV_PB = build_dataset(
    chi2_panth_bao, SECTION_PB, FEATURES_4, RANGES_4,
    n_gaussian=150_000, gaussian_sigma_scale=3.0,
)
print(f'\n=== {TITLE_PBC} ===')
df_pbc, REF_PBC, COV_PBC = build_dataset(
    chi2_panth_bao_cmb, SECTION_PBC, FEATURES_4, RANGES_4,
    n_gaussian=150_000, gaussian_sigma_scale=3.0,
)

## Section 2 — ML: Training & SHAP (all 3 scenarios)

In [ ]:
print(f'=== {TITLE_BAO} ===')
model_bao, info_bao, shap_v_bao, X_s_bao = train_and_shap(df_bao, FEATURES_4, SECTION_BAO, title=TITLE_BAO)

print(f'\n=== {TITLE_PB} ===')
model_pb, info_pb, shap_v_pb, X_s_pb = train_and_shap(df_pb, FEATURES_4, SECTION_PB, title=TITLE_PB)

print(f'\n=== {TITLE_PBC} ===')
model_pbc, info_pbc, shap_v_pbc, X_s_pbc = train_and_shap(df_pbc, FEATURES_4, SECTION_PBC, title=TITLE_PBC)

## Section 3 — ML MCMC (all 3 scenarios)

In [ ]:
samples_ml_bao, ml_meta_bao = run_mcmc_and_getdist(
    model_bao, FEATURES_4, RANGES_4, REF_BAO, SECTION_BAO,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_BAO, proposal_cov=COV_BAO,
    n_trees_mcmc=ML_MCMC_TREES,
)
samples_ml_pb, ml_meta_pb = run_mcmc_and_getdist(
    model_pb, FEATURES_4, RANGES_4, REF_PB, SECTION_PB,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_PB, proposal_cov=COV_PB,
    n_trees_mcmc=ML_MCMC_TREES,
)
samples_ml_pbc, ml_meta_pbc = run_mcmc_and_getdist(
    model_pbc, FEATURES_4, RANGES_4, REF_PBC, SECTION_PBC,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_PBC, proposal_cov=COV_PBC,
    n_trees_mcmc=ML_MCMC_TREES,
)

plot_getdist_comparison(
    [samples_ml_bao, samples_ml_pb, samples_ml_pbc],
    ['BAO only', 'BAO + Pantheon+', 'BAO + Pantheon+ + CMB'],
    FEATURES_4, LABELS,
    markers=MARKERS_DE,
    title='w0waCDM — ML MCMC: 3-scenario comparison',
    save_path=FIGURES_DIR / '7_ml_3way.png',
    filled=[True, True, True], ranges=RANGES_4,
)

## Section 4 — Theory CPU: ProcessPoolExecutor

Pantheon+ scenarios are skipped: multiprocessing with large likelihoods causes IPC/thread issues on GPU servers. Covered by th-np (Section 5) and th-gpu (Section 6).

In [ ]:
samples_cpu_bao, th_cpu_meta_bao = run_theory_mcmc(
    chi2_bao_only, FEATURES_4, RANGES_4, REF_BAO, SECTION_BAO,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_BAO, proposal_cov=COV_BAO,
)
th_cpu_meta_pb  = {'cached': False, 'wall_s': None, 'n_samples': None, 'skipped': True}
th_cpu_meta_pbc = {'cached': False, 'wall_s': None, 'n_samples': None, 'skipped': True}
print('[TH-CPU] panth scenarios: skipped (IPC/thread issues with large likelihoods on GPU server)')

## Section 5 — Theory numpy: batched (all 3 scenarios)

In [ ]:
np_fn_bao = make_chi2_numpy_fn(panth=None,  bao=bao)
np_fn_pb  = make_chi2_numpy_fn(panth=panth, bao=bao)
np_fn_pbc = make_chi2_numpy_fn(panth=panth, bao=bao, planck_prior=True)
print('Numpy predict functions ready.')

In [ ]:
samples_np_bao, th_np_meta_bao = run_theory_mcmc_numpy(
    np_fn_bao, FEATURES_4, RANGES_4, REF_BAO, SECTION_BAO,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_BAO, proposal_cov=COV_BAO,
)
samples_np_pb, th_np_meta_pb = run_theory_mcmc_numpy(
    np_fn_pb, FEATURES_4, RANGES_4, REF_PB, SECTION_PB,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_PB, proposal_cov=COV_PB,
)
samples_np_pbc, th_np_meta_pbc = run_theory_mcmc_numpy(
    np_fn_pbc, FEATURES_4, RANGES_4, REF_PBC, SECTION_PBC,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_PBC, proposal_cov=COV_PBC,
)

plot_getdist_comparison(
    [samples_np_bao, samples_np_pb, samples_np_pbc],
    ['BAO only', 'BAO + Pantheon+', 'BAO + Pantheon+ + CMB'],
    FEATURES_4, LABELS,
    markers=MARKERS_DE,
    title='w0waCDM — Theory numpy MCMC: 3-scenario comparison',
    save_path=FIGURES_DIR / '7_theory_np_3way.png',
    filled=[True, True, True], ranges=RANGES_4,
)

## Section 6 — Theory GPU: JAX (all 3 scenarios)

In [ ]:
try:
    from cosmoml.theory.jax_theory import make_chi2_gpu_fn as _make_chi2_gpu_fn
    _JAX_THEORY_OK = True
    print('JAX/GPU available')
except ImportError:
    _JAX_THEORY_OK = False
    print('JAX not found — theory GPU benchmark will be skipped')

th_gpu_meta_bao = th_gpu_meta_pb = th_gpu_meta_pbc = None
samples_gpu_bao = samples_gpu_pb = samples_gpu_pbc = None

if not _JAX_THEORY_OK:
    print('[!] JAX not available — skipping Theory GPU sections')
else:
    print('Compiling JAX functions (warm-up)...')
    gpu_fn_bao = _make_chi2_gpu_fn(panth=None,  bao=bao)
    gpu_fn_pb  = _make_chi2_gpu_fn(panth=panth, bao=bao)
    gpu_fn_pbc = _make_chi2_gpu_fn(panth=panth, bao=bao, planck_prior=True)
    print('JAX JIT ready.')

    samples_gpu_bao, th_gpu_meta_bao = run_theory_mcmc_gpu(
        gpu_fn_bao, FEATURES_4, RANGES_4, REF_BAO, SECTION_BAO,
        labels=LABELS, markers=MARKERS_DE, title=TITLE_BAO, proposal_cov=COV_BAO,
    )
    samples_gpu_pb, th_gpu_meta_pb = run_theory_mcmc_gpu(
        gpu_fn_pb, FEATURES_4, RANGES_4, REF_PB, SECTION_PB,
        labels=LABELS, markers=MARKERS_DE, title=TITLE_PB, proposal_cov=COV_PB,
    )
    samples_gpu_pbc, th_gpu_meta_pbc = run_theory_mcmc_gpu(
        gpu_fn_pbc, FEATURES_4, RANGES_4, REF_PBC, SECTION_PBC,
        labels=LABELS, markers=MARKERS_DE, title=TITLE_PBC, proposal_cov=COV_PBC,
    )

    plot_getdist_comparison(
        [samples_gpu_bao, samples_gpu_pb, samples_gpu_pbc],
        ['BAO only', 'BAO + Pantheon+', 'BAO + Pantheon+ + CMB'],
        FEATURES_4, LABELS,
        markers=MARKERS_DE,
        title='w0waCDM — Theory GPU MCMC: 3-scenario comparison',
        save_path=FIGURES_DIR / '7_theory_gpu_3way.png',
        filled=[True, True, True], ranges=RANGES_4,
    )

## Section 7 — Dataset generation: GPU vs CPU

In [19]:
ds_gpu_meta_bao = ds_gpu_meta_pb = ds_gpu_meta_pbc = None

if not _JAX_THEORY_OK:
    print('[!] JAX not available — skipping dataset GPU benchmark')
else:
    _, ds_gpu_meta_bao = build_dataset_gpu(
        gpu_fn_bao, chi2_bao_only, SECTION_BAO, FEATURES_4, RANGES_4, REF_BAO,
    )
    _, ds_gpu_meta_pb = build_dataset_gpu(
        gpu_fn_pb, chi2_panth_bao, SECTION_PB, FEATURES_4, RANGES_4, REF_PB,
    )
    _, ds_gpu_meta_pbc = build_dataset_gpu(
        gpu_fn_pbc, chi2_panth_bao_cmb, SECTION_PBC, FEATURES_4, RANGES_4, REF_PBC,
    )

    print('\n' + '-' * 86)
    print(f'{"scenario":<22}{"gpu [s]":>10}{"cpu×5k→extrap":>16}'
          f'{"speedup":>10}{"max|Δχ²|":>12}{"mean|Δχ²|":>12}')
    print('-' * 86)
    for name, m in [
        ('bao_only',      ds_gpu_meta_bao),
        ('panth_bao',     ds_gpu_meta_pb),
        ('panth_bao_cmb', ds_gpu_meta_pbc),
    ]:
        if m is None: print(f'  {name}'); continue
        cmp = m.get('comparison', {})
        gpu_s = f"{m['gpu_wall_s']:.1f}s" if m['gpu_wall_s'] else 'cached'
        cpu_s = f"{m['cpu_sample_s']:.1f}s→{m['cpu_extrap_s']:.0f}s"
        spd   = f"{m['speedup']:.1f}x" if m['speedup'] else '-'
        maxd  = f"{cmp['max_abs_diff']:.4f}" if cmp.get('max_abs_diff') else '-'
        meand = f"{cmp['mean_abs_diff']:.4f}" if cmp.get('mean_abs_diff') else '-'
        print(f'{name:<22}{gpu_s:>10}{cpu_s:>16}{spd:>10}{maxd:>12}{meand:>12}')
    print('-' * 86)


  [DS-GPU]  Evaluando 200,000 pts (JAX GPU batch)...
  [DS-GPU]  0.94s  (213,743 pts/s)
  [DS-CPU]  Timing sample 5,000 pts (23 cores)...
  [DS-CPU]  4.0s / 5,000 pts → extrap full: 160s
  [DS-SPD]  Speedup GPU vs CPU (extrap): 170.9x
  [DS-CMP]  max|Δχ²|=569.0738 | mean|Δχ²|=3.6707
  [DS-GPU]  Evaluando 200,000 pts (JAX GPU batch)...
  [DS-GPU]  65.85s  (3,037 pts/s)
  [DS-CPU]  Timing sample 5,000 pts (23 cores)...
  [DS-CPU]  110.1s / 5,000 pts → extrap full: 4402s
  [DS-SPD]  Speedup GPU vs CPU (extrap): 66.8x
  [DS-GPU]  Evaluando 200,000 pts (JAX GPU batch)...
  [DS-GPU]  1.09s  (183,513 pts/s)
  [DS-CPU]  Timing sample 5,000 pts (23 cores)...
  [DS-CPU]  107.5s / 5,000 pts → extrap full: 4301s
  [DS-SPD]  Speedup GPU vs CPU (extrap): 3946.2x

--------------------------------------------------------------------------------------
scenario                 gpu [s]   cpu×5k→extrap   speedup    max|Δχ²|   mean|Δχ²|
----------------------------------------------------------------------

## Timings — summary table & JSON

In [ ]:
def _speedup(ml_m, th_m):
    w_ml = (ml_m or {}).get('wall_s')
    w_th = (th_m or {}).get('wall_s')
    return round(w_th / w_ml, 2) if (w_ml and w_th) else None

def _skip(m):
    return m if m is not None else {'wall_s': None, 'n_samples': None, 'cached': True}

timings = {
    'ml_engine':         'RWMH 1024 chains, XGBoost GPU booster (LogChi2)',
    'theory_cpu_engine': 'RWMH 1024 chains, ProcessPoolExecutor exact chi2',
    'theory_np_engine':  'RWMH 1024 chains, numpy batched exact chi2',
    'theory_gpu_engine': 'RWMH 1024 chains, JAX vmap GPU exact chi2',
    'common': {'n_chains': 1024, 'n_steps_cap': 10_000, 'burn_in': 500, 'seed': 42, 'ess_target': 10_000},
    'runs': {
        'bao_only':      {'ml': ml_meta_bao,  'theory_cpu': th_cpu_meta_bao,  'theory_np': th_np_meta_bao,  'theory_gpu': _skip(th_gpu_meta_bao),  'speedup_np_vs_ml': _speedup(ml_meta_bao,  th_np_meta_bao),  'speedup_gpu_vs_ml': _speedup(ml_meta_bao,  _skip(th_gpu_meta_bao))},
        'panth_bao':     {'ml': ml_meta_pb,   'theory_cpu': th_cpu_meta_pb,   'theory_np': th_np_meta_pb,   'theory_gpu': _skip(th_gpu_meta_pb),   'speedup_np_vs_ml': _speedup(ml_meta_pb,   th_np_meta_pb),   'speedup_gpu_vs_ml': _speedup(ml_meta_pb,   _skip(th_gpu_meta_pb))},
        'panth_bao_cmb': {'ml': ml_meta_pbc,  'theory_cpu': th_cpu_meta_pbc,  'theory_np': th_np_meta_pbc,  'theory_gpu': _skip(th_gpu_meta_pbc),  'speedup_np_vs_ml': _speedup(ml_meta_pbc,  th_np_meta_pbc),  'speedup_gpu_vs_ml': _speedup(ml_meta_pbc,  _skip(th_gpu_meta_pbc))},
    },
}

timings_path = PAPER_DIR / 'timings.json'
with open(timings_path, 'w') as f:
    json.dump(timings, f, indent=2)
print(f'Timings saved: {timings_path}\n')

def _fs(m):
    if m is None: return '-'
    if m.get('skipped'): return 'skipped'
    v = m.get('wall_s')
    return f'{v:.1f}s' if v is not None else '-'
def _fsp(v): return f'{v:.1f}x' if v is not None else '-'

rows = [
    ('bao_only',      ml_meta_bao,  th_cpu_meta_bao,  th_np_meta_bao,  th_gpu_meta_bao),
    ('panth_bao',     ml_meta_pb,   th_cpu_meta_pb,   th_np_meta_pb,   th_gpu_meta_pb),
    ('panth_bao_cmb', ml_meta_pbc,  th_cpu_meta_pbc,  th_np_meta_pbc,  th_gpu_meta_pbc),
]
print('-' * 100)
print(f'{"section":<18}{"ml [s]":>12}{"th-cpu [s]":>12}{"th-np [s]":>12}{"th-gpu [s]":>12}{"×np/ml":>10}{"×gpu/ml":>10}')
print('-' * 100)
for name, ml_m, cpu_m, np_m, tg_m in rows:
    tg = _skip(tg_m)
    print(f'{name:<18}{_fs(ml_m):>12}{_fs(cpu_m):>12}{_fs(np_m):>12}{_fs(tg):>12}{_fsp(_speedup(ml_m, np_m)):>10}{_fsp(_speedup(ml_m, tg)):>10}')
print('-' * 100)